# 🔍 Seeing Through Deepfakes — Integration Pipeline

This notebook combines **three detection modules** into a single interactive pipeline:

| Module | Role |
|---|---|
| **Metadata Module** | Extracts EXIF / C2PA / binary markers and scores P(AI) from file metadata |
| **Visual Classifier** | ViT-based image classifier (fine-tuned) that scores P(AI) from pixel content |
| **Deepfake Module** | Face detection, scene classification, and landmark retrieval — triggered only for AI-positive images |

A **toggleable decision formula** fuses the first two scores into a single verdict.  
Three formulas are provided: *Weighted Average*, *Conservative Threshold (AND-gate)*, and *Bayesian Fusion*.

## 1. Setup & Imports

In [1]:
import sys
import os
import io
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add project root to sys.path so we can import from src.*
PROJECT_ROOT = str(Path.cwd().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from PIL import Image
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Our modules
from src.metadata_module import analyse_image, AnalysisResult
from src.visual_module.visual_classifier import VisualClassifier
from src.deepfake_module.deepfake_classifier import DeepfakeClassifier
from src.integration_pipeline.fusion import (
    get_fusion_strategy,
    extract_visual_ai_probability,
    AVAILABLE_STRATEGIES,
    FusionResult,
)

print("✅ All imports successful.")

W0529 12:13:49.905000 77729 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


✅ All imports successful.


## 2. Load Models

This cell loads all three models into memory. It may take a minute the first time (weights are downloaded from HuggingFace).

In [2]:
# --- Visual Classifier (base + fine-tuned delta) ---
VISUAL_DELTA_PATH = os.path.join(PROJECT_ROOT, "src", "visual_module", "fine_tuned_model_delta", "run_02_ft_genimage_w_julienlucas_weight_delta.pt")

print("Loading Visual Classifier (base + fine-tuned delta)...")
visual_classifier = VisualClassifier(
    model_name_or_path="dima806/ai_vs_human_generated_image_detection",
    delta_path=VISUAL_DELTA_PATH,
)
print("✅ Visual Classifier ready.\n")

# --- Deepfake Classifier ---
DEEPFAKE_INDEX_PATH = os.path.join(PROJECT_ROOT, "src", "deepfake_module", "models", "landmarks_index.faiss")
DEEPFAKE_META_PATH  = os.path.join(PROJECT_ROOT, "src", "deepfake_module", "models", "landmarks_metadata.json")

print("Loading Deepfake Classifier (MTCNN + Places365 + DINOv2 landmark retrieval)...")
deepfake_classifier = DeepfakeClassifier(
    index_path=DEEPFAKE_INDEX_PATH,
    metadata_path=DEEPFAKE_META_PATH,
)
print("✅ Deepfake Classifier ready.")

Loading Visual Classifier (base + fine-tuned delta)...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Applying weight delta from '/Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters/Proof-of-Concept/Seeing-Through-Deepfakes/src/visual_module/fine_tuned_model_delta/run_02_ft_genimage_w_julienlucas_weight_delta.pt'...
Delta applied successfully.
✅ Visual Classifier ready.

Loading Deepfake Classifier (MTCNN + Places365 + DINOv2 landmark retrieval)...
Loading Face Detection model: MTCNN
Loading Scene model: birder-project/rope_vit_reg4_b14_capi-places365


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading Landmark Retrieval model: facebook/dinov2-base


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading FAISS index from /Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters/Proof-of-Concept/Seeing-Through-Deepfakes/src/deepfake_module/models/landmarks_index.faiss...
Landmark index loaded successfully.
✅ Deepfake Classifier ready.


## 3. Configuration Panel

Select the **fusion formula** and tune its parameters with the sliders below.  
Changes take effect on the next image upload — no need to re-run this cell.

In [3]:
# ── Fusion formula selector ──────────────────────────────────────────────
formula_dropdown = widgets.Dropdown(
    options=[
        ("Weighted Average", "weighted_average"),
        ("Conservative Threshold (AND-gate)", "conservative_threshold"),
        ("Bayesian Fusion", "bayesian"),
    ],
    value="weighted_average",
    description="Formula:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)

# ── Weighted Average parameters ─────────────────────────────────────────
w_meta_slider = widgets.FloatSlider(
    value=0.3, min=0.0, max=1.0, step=0.05,
    description="W_meta:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
w_visual_slider = widgets.FloatSlider(
    value=0.7, min=0.0, max=1.0, step=0.05,
    description="W_visual:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
meta_accuracy_slider = widgets.FloatSlider(
    value=0.70, min=0.5, max=1.0, step=0.01,
    description="Meta Acc.:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
visual_accuracy_slider = widgets.FloatSlider(
    value=0.83, min=0.5, max=1.0, step=0.01,
    description="Vis. Acc.:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
face_padding_slider = widgets.FloatSlider(
    value=0.30, min=0.0, max=1.0, step=0.05,
    description="Face Pad.:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
wa_threshold_slider = widgets.FloatSlider(
    value=0.55, min=0.1, max=0.95, step=0.05,
    description="Threshold:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)

wa_box = widgets.VBox([
    widgets.HTML("<b>Weighted Average Parameters</b>"),
    w_meta_slider, w_visual_slider,
    meta_accuracy_slider, visual_accuracy_slider,
    face_padding_slider,
    wa_threshold_slider,
    widgets.HTML("<small><i>Metadata is down-weighted because it can be easily manipulated. "
                 "Each module's effective weight = nominal weight x accuracy. "
                 "If a face is detected, the visual score is enhanced using max(whole_image, crop).</i></small>"),
])

# ── Conservative Threshold parameters ───────────────────────────────────
ct_meta_thresh = widgets.FloatSlider(
    value=0.70, min=0.1, max=0.99, step=0.05,
    description="Meta thr.:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
ct_visual_thresh = widgets.FloatSlider(
    value=0.65, min=0.1, max=0.99, step=0.05,
    description="Vis. thr.:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
ct_box = widgets.VBox([
    widgets.HTML("<b>Conservative Threshold Parameters</b>"),
    ct_meta_thresh, ct_visual_thresh,
    widgets.HTML("<small><i>AND-gate: both modules must exceed their thresholds. "
                 "This avoids false positives from metadata always scoring 0.99 on AI tags.</i></small>"),
])

# ── Bayesian Fusion parameters ──────────────────────────────────────────
bayes_prior = widgets.FloatSlider(
    value=0.50, min=0.05, max=0.95, step=0.05,
    description="Prior:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
bayes_threshold = widgets.FloatSlider(
    value=0.55, min=0.1, max=0.95, step=0.05,
    description="Threshold:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
bayes_box = widgets.VBox([
    widgets.HTML("<b>Bayesian Fusion Parameters</b>"),
    bayes_prior, bayes_threshold,
    widgets.HTML("<small><i>Prior = your belief that any random image is AI before seeing evidence. "
                 "0.5 = no bias. Each module's score acts as independent evidence that is combined "
                 "via Bayes' rule.</i></small>"),
])

# ── Secondary deepfake threshold ────────────────────────────────────────
deepfake_threshold_slider = widgets.FloatSlider(
    value=0.50, min=0.1, max=0.95, step=0.05,
    description="DF thr.:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)
df_box = widgets.VBox([
    widgets.HTML("<b>Deepfake Sub-threshold</b>"),
    deepfake_threshold_slider,
    widgets.HTML("<small><i>For AI-positive images: if the deepfake module detects a face or landmark "
                 "above this threshold → 'Probable Deepfake'; otherwise → 'AI-generated, "
                 "not necessarily a deepfake'.</i></small>"),
])

# ── Dynamic visibility ──────────────────────────────────────────────────
param_stack = widgets.VBox()

def update_param_visibility(change=None):
    formula = formula_dropdown.value
    if formula == "weighted_average":
        param_stack.children = [wa_box, df_box]
    elif formula == "conservative_threshold":
        param_stack.children = [ct_box, df_box]
    elif formula == "bayesian":
        param_stack.children = [bayes_box, df_box]

formula_dropdown.observe(update_param_visibility, names="value")
update_param_visibility()  # initialise

config_panel = widgets.VBox([
    widgets.HTML("<h3>⚙️ Pipeline Configuration</h3>"),
    formula_dropdown,
    param_stack,
])

display(config_panel)

## 4. Image Upload & Pipeline Execution

Upload an image below.  The full pipeline runs automatically on upload:  
**Metadata → Visual → Fusion → (if AI) Deepfake → Verdict + Explainability**

In [4]:
# ── Upload widget ───────────────────────────────────────────────────────
upload_widget = widgets.FileUpload(
    accept=".png,.jpg,.jpeg,.webp",
    multiple=False,
    description="Upload Image",
    layout=widgets.Layout(width="300px"),
)

output_area = widgets.Output()


# ── Helpers ──────────────────────────────────────────────────────────────

def _build_strategy():
    """Read the current widget values and return a configured FusionStrategy."""
    formula = formula_dropdown.value
    if formula == "weighted_average":
        return get_fusion_strategy(
            "weighted_average",
            w_meta=w_meta_slider.value,
            w_visual=w_visual_slider.value,
            decision_threshold=wa_threshold_slider.value,
            meta_accuracy=meta_accuracy_slider.value,
            visual_accuracy=visual_accuracy_slider.value,
        )
    elif formula == "conservative_threshold":
        return get_fusion_strategy(
            "conservative_threshold",
            meta_threshold=ct_meta_thresh.value,
            visual_threshold=ct_visual_thresh.value,
        )
    elif formula == "bayesian":
        return get_fusion_strategy(
            "bayesian",
            prior=bayes_prior.value,
            decision_threshold=bayes_threshold.value,
        )


def _colour_for_verdict(verdict: str) -> str:
    """Return a CSS colour string based on the verdict."""
    v = verdict.lower()
    if "deepfake" in v:
        return "#e74c3c"   # red
    if "ai" in v or "generated" in v:
        return "#f39c12"   # orange
    return "#27ae60"       # green


def _pil_to_base64_img_tag(img: Image.Image, max_size: int = 150) -> str:
    import base64
    from io import BytesIO
    buffered = BytesIO()
    thumb = img.copy()
    thumb.thumbnail((max_size, max_size))
    thumb.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return f'<img src="data:image/jpeg;base64,{img_str}" style="border: 2px solid #ccc; border-radius: 8px; margin: 5px 0; max-height: {max_size}px;" />'


def _render_results(
    pil_image: Image.Image,
    meta_result: AnalysisResult,
    visual_result: dict,
    cropped_visual_result: dict | None,
    cropped_face: Image.Image | None,
    fusion_result: FusionResult,
    deepfake_result: dict | None,
    verdict: str,
):
    """Render a rich HTML results panel inside the output_area."""

    colour = _colour_for_verdict(verdict)

    # ── Verdict banner ───────────────────────────────────────────────
    html = f"""
    <div style="border:2px solid {colour}; border-radius:12px; padding:18px;
                margin:10px 0; background:linear-gradient(135deg, {colour}22, {colour}08);">
      <h2 style="margin:0 0 6px 0; color:{colour};">🏷️ {verdict}</h2>
      <p style="margin:0; font-size:14px; color:#555;">Fused AI probability: <b>{fusion_result.ai_probability:.2%}</b>
         &nbsp;|&nbsp; Formula: <b>{fusion_result.formula_name}</b></p>
    </div>
    """

    # ── Confidence breakdown table ───────────────────────────────────
    visual_ai_prob = extract_visual_ai_probability(visual_result)
    
    html += """
    <h3>📊 Confidence Breakdown</h3>
    <table style="border-collapse:collapse; width:100%; max-width:700px; font-size:14px;">
      <thead>
        <tr style="background:#f0f0f0;">
          <th style="padding:8px; text-align:left; border-bottom:2px solid #ccc;">Module</th>
          <th style="padding:8px; text-align:right; border-bottom:2px solid #ccc;">AI Probability</th>
          <th style="padding:8px; text-align:left; border-bottom:2px solid #ccc;">Decision</th>
        </tr>
      </thead>
      <tbody>
    """
    
    html += f"""
        <tr>
          <td style="padding:6px 8px;">Metadata Module</td>
          <td style="padding:6px 8px; text-align:right;">{meta_result.ai_probability:.2%}</td>
          <td style="padding:6px 8px;">{meta_result.decision}</td>
        </tr>
        <tr style="background:#fafafa;">
          <td style="padding:6px 8px;">Visual Classifier (Whole Image)</td>
          <td style="padding:6px 8px; text-align:right;">{visual_ai_prob:.2%}</td>
          <td style="padding:6px 8px;">{visual_result['prediction']} (conf: {visual_result['confidence']:.2%})</td>
        </tr>
    """
    
    if cropped_visual_result is not None:
        cropped_ai_prob = extract_visual_ai_probability(cropped_visual_result)
        html += f"""
        <tr>
          <td style="padding:6px 8px;">Visual Classifier (Cropped Face)</td>
          <td style="padding:6px 8px; text-align:right;">{cropped_ai_prob:.2%}</td>
          <td style="padding:6px 8px;">{cropped_visual_result['prediction']} (conf: {cropped_visual_result['confidence']:.2%})</td>
        </tr>
        """
        
    html += f"""
        <tr style="font-weight:bold; background:#f0f8ff;">
          <td style="padding:6px 8px; border-top:2px solid #ccc;">Fused ({fusion_result.formula_name})</td>
          <td style="padding:6px 8px; text-align:right; border-top:2px solid #ccc;">{fusion_result.ai_probability:.2%}</td>
          <td style="padding:6px 8px; border-top:2px solid #ccc;">{'AI-Generated' if fusion_result.is_ai else 'Real'}</td>
        </tr>
      </tbody>
    </table>
    """

    # ── Fusion formula details ───────────────────────────────────────
    html += "<h3>🧮 Fusion Formula Details</h3><ul style='font-size:13px;'>"
    for k, v in fusion_result.explanation.items():
        html += f"<li><b>{k}</b>: {v}</li>"
    html += "</ul>"

    # ── Metadata explainability ──────────────────────────────────────
    html += "<h3>📋 Metadata Module — Explainability</h3>"
    html += "<p style='font-size:13px;'><b>Rationale:</b></p><ul style='font-size:13px;'>"
    for r in meta_result.rationale:
        html += f"<li>{r}</li>"
    if not meta_result.rationale:
        html += "<li><i>No specific rationale entries.</i></li>"
    html += "</ul>"

    features = meta_result.features
    html += "<p style='font-size:13px;'><b>Feature Highlights:</b></p>"
    html += "<table style='border-collapse:collapse; font-size:13px; max-width:500px;'>"
    feature_items = [
        ("C2PA / Provenance", features.has_c2pa),
        ("AI Claim in metadata", features.has_ai_claim),
        ("Camera Make", features.has_make),
        ("Camera Model", features.has_model),
        ("Lens Model", features.has_lens_model),
        ("MakerNote", features.has_makernote),
        ("GPS Data", features.has_gps),
        ("Camera Claim", features.has_camera_claim),
        ("Edit Claim", features.has_edit_claim),
        ("Suspicious (SW tags, no camera)", features.suspicious_only_software_tags),
        ("Suspicious (perfect timestamp)", features.suspicious_perfect_timestamp),
    ]
    for label, val in feature_items:
        icon = "✅" if val else "❌"
        html += f"<tr><td style='padding:3px 8px;'>{label}</td><td style='padding:3px 8px;'>{icon}</td></tr>"
    html += "</table>"

    if features.keyword_hits:
        html += f"<p style='font-size:13px;'><b>Keyword hits:</b> {', '.join(features.keyword_hits)}</p>"
    if features.binary_hits:
        html += f"<p style='font-size:13px;'><b>Binary marker hits:</b> {', '.join(features.binary_hits)}</p>"

    # ── Visual Classifier explainability ─────────────────────────────
    html += "<h3>🖼️ Visual Classifier (Whole Image) — Explainability</h3>"
    all_scores = visual_result.get("all_scores", {})
    if all_scores:
        html += "<table style='border-collapse:collapse; font-size:13px; max-width:400px;'>"
        html += "<tr style='background:#f0f0f0;'><th style='padding:4px 8px; text-align:left;'>Class</th>"
        html += "<th style='padding:4px 8px; text-align:right;'>Score</th></tr>"
        for cls_name, score in sorted(all_scores.items(), key=lambda x: -x[1]):
            html += f"<tr><td style='padding:3px 8px;'>{cls_name}</td>"
            html += f"<td style='padding:3px 8px; text-align:right;'>{score:.4f}</td></tr>"
        html += "</table>"

    # ── Cropped Face Classifier explainability ────────────────────────
    if cropped_face is not None and cropped_visual_result is not None:
        html += "<h3>👤 Cropped Face Classifier — Explainability</h3>"
        html += "<div style='display: flex; align-items: center; gap: 15px; margin-bottom: 15px;'>"
        html += f"  <div>{_pil_to_base64_img_tag(cropped_face, 120)}</div>"
        html += "  <div>"
        cropped_ai_prob = extract_visual_ai_probability(cropped_visual_result)
        html += f"    <p style='margin:0; font-size:13px;'><b>Prediction:</b> {cropped_visual_result['prediction']}</p>"
        html += f"    <p style='margin:0; font-size:13px;'><b>AI Probability:</b> {cropped_ai_prob:.2%}</p>"
        html += f"    <p style='margin:0; font-size:13px;'><b>Confidence:</b> {cropped_visual_result['confidence']:.2%}</p>"
        html += "  </div>"
        html += "</div>"
        
        all_cropped_scores = cropped_visual_result.get("all_scores", {})
        if all_cropped_scores:
            html += "<table style='border-collapse:collapse; font-size:13px; max-width:400px;'>"
            html += "<tr style='background:#f0f0f0;'><th style='padding:4px 8px; text-align:left;'>Class</th>"
            html += "<th style='padding:4px 8px; text-align:right;'>Score</th></tr>"
            for cls_name, score in sorted(all_cropped_scores.items(), key=lambda x: -x[1]):
                html += f"<tr><td style='padding:3px 8px;'>{cls_name}</td>"
                html += f"<td style='padding:3px 8px; text-align:right;'>{score:.4f}</td></tr>"
            html += "</table>"

    # ── Deepfake Module explainability (if run) ──────────────────────
    if deepfake_result is not None:
        html += "<h3>🎭 Deepfake Module — Explainability</h3>"
        da = deepfake_result.get("deepfake_analysis")
        if da:
            # Face
            face = da.get("face_analysis", {})
            html += f"<p style='font-size:13px;'><b>Face Detection:</b> {face.get('label', 'N/A')} "
            html += f"(confidence: {face.get('confidence', 0):.2%})</p>"
            # Scene
            scene = da.get("scene_analysis", {})
            html += f"<p style='font-size:13px;'><b>Scene Classification:</b> {scene.get('label', 'N/A')} "
            html += f"(confidence: {scene.get('confidence', 0):.2%})</p>"
            # Landmark
            landmark = da.get("landmark_analysis", {})
            html += f"<p style='font-size:13px;'><b>Landmark Retrieval:</b> {landmark.get('label', 'N/A')} "
            html += f"(confidence: {landmark.get('confidence', 0):.4f})</p>"
            if landmark.get("all_matches"):
                html += "<details><summary style='font-size:13px; cursor:pointer;'>Show all landmark matches</summary><ul style='font-size:12px;'>"
                for name, sims in landmark["all_matches"].items():
                    avg = sum(sims) / len(sims)
                    html += f"<li>{name}: avg sim = {avg:.4f} ({len(sims)} matches)</li>"
                html += "</ul></details>"

            html += f"<p style='font-size:13px;'><b>Has Face:</b> {'✅' if da.get('has_face') else '❌'}"
            html += f" &nbsp;|&nbsp; <b>Has Landmark:</b> {'✅' if da.get('has_place') else '❌'}</p>"

        final = deepfake_result.get("final_decision", "N/A")
        html += f"<p style='font-size:13px;'><b>Deepfake Module Decision:</b> {final}</p>"

    display(HTML(html))


# ── Main pipeline callback ──────────────────────────────────────────────

def run_pipeline(change):
    """Triggered automatically when a file is uploaded."""
    uploaded_files = upload_widget.value
    if not uploaded_files:
        return

    with output_area:
        clear_output(wait=True)

        # Get the uploaded file content (ipywidgets v8+: value is a tuple of dicts)
        file_info = uploaded_files[0]
        file_content = file_info["content"]
        file_name = file_info["name"]

        print(f"📁 Processing: {file_name}")
        print("─" * 60)

        # Load as PIL Image
        pil_image = Image.open(io.BytesIO(file_content))
        if pil_image.mode != "RGB":
            pil_image = pil_image.convert("RGB")

        # Display the uploaded image
        thumb = pil_image.copy()
        thumb.thumbnail((400, 400))
        display(thumb)

        # ── Step 1: Metadata Module ──────────────────────────────────
        print("\n🔬 Running Metadata Module...")
        # Write to a temp file so exiftool can read the raw bytes (preserves EXIF)
        suffix = Path(file_name).suffix or ".png"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp.write(file_content)
            tmp_path = tmp.name

        try:
            meta_result = analyse_image(tmp_path)
        finally:
            os.unlink(tmp_path)  # clean up

        print(f"   Metadata AI probability: {meta_result.ai_probability:.2%}")
        print(f"   Metadata decision:       {meta_result.decision}")

        # ── Step 2: Visual Classifier ────────────────────────────────
        print("\n🖼️  Running Visual Classifier (Whole Image)...")
        visual_result = visual_classifier.predict(pil_image)
        visual_ai_prob = extract_visual_ai_probability(visual_result)
        print(f"   Visual prediction:  {visual_result['prediction']}")
        print(f"   Visual AI prob:     {visual_ai_prob:.2%}")
        print(f"   Visual confidence:  {visual_result['confidence']:.2%}")

        # ── Step 3: Face crop detection & classification (unconditional) ──
        print("\n👤 Running Face Detection...")
        face_res = deepfake_classifier.predict_face(pil_image)
        bbox = face_res.get("bbox")
        
        cropped_visual_result = None
        cropped_visual_ai_prob = None
        cropped_face = None
        if bbox is not None:
            print("   Face detected! Cropping and re-classifying...")
            from src.integration_pipeline.fusion import crop_face_region
            cropped_face = crop_face_region(pil_image, bbox, padding=face_padding_slider.value)
            cropped_visual_result = visual_classifier.predict(cropped_face)
            cropped_visual_ai_prob = extract_visual_ai_probability(cropped_visual_result)
            print(f"   Cropped Face AI prob: {cropped_visual_ai_prob:.2%}")
        else:
            print("   No face detected. Skipping cropped face analysis.")

        # ── Step 4: Fusion ───────────────────────────────────────────
        print(f"\n🧮 Fusing with: {formula_dropdown.label}...")
        strategy = _build_strategy()
        fusion_result = strategy.fuse(
            metadata_ai_prob=meta_result.ai_probability,
            visual_ai_prob=visual_ai_prob,
            cropped_visual_ai_prob=cropped_visual_ai_prob,
        )
        print(f"   Fused AI probability: {fusion_result.ai_probability:.2%}")
        print(f"   Fused decision:       {'AI-Generated' if fusion_result.is_ai else 'Real'}")

        # ── Step 5: Conditional Deepfake Analysis ────────────────────
        deepfake_result = None
        if fusion_result.is_ai:
            print("\n🎭 AI-positive — Running Deepfake Module...")
            deepfake_result = deepfake_classifier.predict(pil_image)

            da = deepfake_result.get("deepfake_analysis")
            
            # Read the deepfake sub-threshold from the widget
            deepfake_threshold = deepfake_threshold_slider.value
            
            has_face = da.get("has_face", False) if da else False
            is_face_deepfake = has_face and cropped_visual_ai_prob is not None and cropped_visual_ai_prob >= deepfake_threshold

            has_place = da.get("has_place", False) if da else False
            is_place_deepfake = has_place and da.get("landmark_analysis", {}).get("confidence", 0.0) >= deepfake_threshold

            if is_face_deepfake or is_place_deepfake:
                verdict = "🚨 Probable Deepfake (AI-generated image containing identifiable face/place)"
            else:
                verdict = "⚠️ Probably AI-generated, but not necessarily a deepfake (no face or landmark detected)"

            print(f"   Face detected: {has_face} (deepfake: {is_face_deepfake})")
            print(f"   Landmark detected: {has_place} (deepfake: {is_place_deepfake})")
        else:
            verdict = f"✅ Probably Real (confidence: {1 - fusion_result.ai_probability:.2%})"

        print(f"\n{'═' * 60}")
        print(f"VERDICT: {verdict}")
        print(f"{'═' * 60}\n")

        # ── Render rich HTML results ─────────────────────────────────
        _render_results(
            pil_image=pil_image,
            meta_result=meta_result,
            visual_result=visual_result,
            cropped_visual_result=cropped_visual_result,
            cropped_face=cropped_face,
            fusion_result=fusion_result,
            deepfake_result=deepfake_result,
            verdict=verdict,
        )


# ── Wire up the callback ────────────────────────────────────────────────
upload_widget.observe(run_pipeline, names="value")

display(widgets.VBox([
    widgets.HTML("<h3>📸 Upload an Image</h3>"),
    upload_widget,
    output_area,
]))

## 5. Launch ChatGPT-Style Browser Interface

Run the cell below to start a background web server and automatically open a beautiful ChatGPT-style interface in your browser. Any uploads and configuration updates made in the browser will use the models already loaded in this notebook.

In [5]:
from src.integration_pipeline.app import start_server_thread

# Start the web server and open the browser window
start_server_thread(visual_classifier, deepfake_classifier);


🚀 Seeing through Deepfakes web interface is live!
🔗 URL: http://127.0.0.1:5001
Opening browser window automatically...
